# 00 · Setup

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

The [sc-rnaseq-wf](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/README.md) WDL workflow runs a harmonized first pass over ASAP PMDBS sc/sn RNA-seq data: per-sample preprocessing (Cell Ranger → CellBender → AnnData) followed by a **cohort analysis** (merge → QC filter → MapMyCells → normalize → scVI/scANVI → clustering → Harmony → scib → plots).

These notebooks re-run the cohort analysis one step at a time for a single team's dataset, the way the workflow runs a per-project cohort analysis (`cohort_id = asap_team_id = team-jakobsson`). Each notebook:

1. explains what the step does and why, from the script source,
2. runs the same script through its [sc-tools](https://github.com/ASAP-CRN/sc-tools) console command (`sc-<script>`), with the workflow's default parameters,
3. loads the outputs and compares them with the curated release where possible.

| Notebook | Step | WDL task |
| :- | :- | :- |
| [00_setup](00_setup.ipynb) | Workspace, sc-tools and references | - |
| [01_prep_metadata](01_prep_metadata.ipynb) | Per-sample QC metrics and doublet scores | `preprocess.counts_to_adata` |
| [02_merge_and_plot_qc](02_merge_and_plot_qc.ipynb) | Merge samples, plot QC metrics | `merge_and_plot_qc_metrics` |
| [03_filter](03_filter.ipynb) | QC filtering | `filter` |
| [04_mmc](04_mmc.ipynb) | MapMyCells cell type mapping | `map_cell_types` |
| [05_process](05_process.ipynb) | Normalize, cell cycle, HVGs, PCA | `normalize` |
| [06_transcriptional_phenotype](06_transcriptional_phenotype.ipynb) | Attach MMC labels | `add_mapped_cell_types` |
| [07_integrate_scvi](07_integrate_scvi.ipynb) | scVI integration | `cluster_data.integrate_sample_data` |
| [08_clustering_umap](08_clustering_umap.ipynb) | Neighbors, Leiden, UMAP | `cluster_data.cluster_cells` |
| [09_annotate](09_annotate.ipynb) | scANVI labels + Harmony + scib | `assign_remaining_cells` + `integrate_harmony` |
| [10_artifact_metrics](10_artifact_metrics.ipynb) | Integration metrics (scib) | `artifact_metrics` |
| [11_plot_groups_and_feats](11_plot_groups_and_feats.ipynb) | UMAP plots of groups and features | `plot_groups_and_features` |

**Starting point.** The per-sample `*.cleaned_unfiltered.h5ad` files in the curated bucket's `preprocess/` folder: CellBender-cleaned counts with QC metrics and doublet scores already added. Cell Ranger and CellBender are not re-run.

**Not covered.** The optional Kamath/Siletti reference spike-in (`prep_da_spike_in`) and downsampling (`n_cells`) are skipped; all cells are kept.

**sc-tools vs. the workflow.** sc-tools packages the workflow's scripts, unchanged, as `sc-*` commands (`sc_tools.__upstream_commit__` records the sc-rnaseq-wf commit). It adds `sc-annotate`, which runs scANVI labeling, Harmony and the scib metrics in one step (notebook 09), and a check in the plotting step for missing UMAP groups.

**Compute.** Notebooks 01–06 run on CPU (04, MapMyCells, needs a lot of memory). Notebooks 07–09 need a GPU: scVI/scANVI training, and `rapids-singlecell` for clustering, UMAP and Harmony.

All paths and parameters are in [`wf_config.py`](wf_config.py).

## 1. Workspace orientation

On Verily Workbench (VWB) the workspace resources are mounted under `~/workspace` (see the [ASAP CRN Learning Lab](https://github.com/ASAP-CRN/asap-crn-learning-lab)):

- `~/workspace/01_PMDBS_scRNAseq/<bucket>/pmdbs_sc_rnaseq/` – curated workflow outputs (`preprocess/`, `cohort_analysis/`)
- `~/workspace/release_resources/<dataset>/metadata/` – release metadata tables
- `~/workspace/ws_files/` – your scratch space; these notebooks write to `ws_files/sc_rnaseq_wf/team-jakobsson/`

Off VWB, set the `ASAP_WS_ROOT` environment variable to a directory with the same layout before starting Jupyter.

In [ ]:
import wf_config as cfg
import pandas as pd

if not cfg.WS_ROOT.exists():
    print(f"{cfg.WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount

for name in ["WS_ROOT", "WS_FILES", "DATASET_PATH", "PREPROCESS_PATH", "CURATED_COHORT_PATH", "METADATA_PATH", "RUN_DIR", "REFERENCES_PATH"]:
    path = getattr(cfg, name)
    print(f"{name:20s} {'ok     ' if path.exists() else 'missing'} {path}")

## 2. The dataset

The curated bucket name is built from the dataset ID, `{team-xxyy}-{source}-{modality_flavour}-{context}`:

| Component | Value |
| :- | :- |
| `workflow_name` | `pmdbs_sc_rnaseq` |
| `team_id` (= `cohort_id`) | `team-jakobsson` |
| `dataset_id` | `team-jakobsson-pmdbs-sn-rnaseq` |
| bucket | `asap-curated-team-jakobsson-pmdbs-sn-rnaseq` |

`preprocess/` holds one set of files per sample, named `${dataset_id}.${sample_id}.*`; `cohort_analysis/` holds the team-level cohort analysis we are re-creating.

In [ ]:
cleaned = sorted(cfg.PREPROCESS_PATH.glob("*.cleaned_unfiltered.h5ad"))
print(f"{len(cleaned)} per-sample cleaned_unfiltered.h5ad files")
pd.DataFrame(
    {"file": [p.name for p in cleaned], "size_MB": [round(p.stat().st_size / 1e6, 1) for p in cleaned]}
)

In [ ]:
print("Curated cohort_analysis outputs:")
for p in sorted(cfg.CURATED_COHORT_PATH.glob("*")):
    print(" -", p.name)

In [ ]:
# The sample list written by the workflow's write_cohort_sample_list task
sample_list = cfg.curated("sample_list.tsv")
if sample_list.exists():
    display(pd.read_csv(sample_list, sep="\t"))
else:
    print(f"{sample_list.name} not found")

## 3. Install sc-tools

Each workflow script is a console command, `sc-<script-name>`, taking the same arguments as in the Docker image. Install it with the MapMyCells extra. For a local checkout with unpushed changes (e.g. `sc-annotate`), use an editable install instead.

scVI/scANVI need GPU builds of `torch` and `jax`; see `cuda-env.yml` in the sc-tools repo.

In [ ]:
%pip install --quiet "sc-tools[mmc] @ git+https://github.com/ASAP-CRN/sc-tools.git"
# or, from a local checkout:
# %pip install --quiet -e "../../../sc-tools[mmc]"

In [ ]:
import shutil
import sc_tools

print("sc-tools", sc_tools.__version__, "synced from sc-rnaseq-wf @", sc_tools.__upstream_commit__[:12])
commands = [
    "sc-prep-metadata", "sc-merge-and-plot-qc", "sc-filter", "sc-mmc", "sc-process",
    "sc-transcriptional-phenotype", "sc-integrate-scvi", "sc-clustering-umap", "sc-annotate",
    "sc-artifact-metrics", "sc-plot-groups-and-feats",
]
missing = [c for c in commands if shutil.which(c) is None]
print("all commands found" if not missing else f"missing commands: {missing}")

## 4. MapMyCells reference

MapMyCells maps each cell to an Allen Brain Cell Atlas taxonomy using a *precomputed stats* file (average expression per taxonomy node). The workflow's human options are:

| `mmc_taxonomy` | Reference | File |
| :- | :- | :- |
| `Siletti` | 10x whole human brain (WHB-10Xv3, CCN20240330) | `precomputed_stats.siletti.training.h5` |
| `SEAAD` | SEA-AD middle temporal gyrus | `precomputed_stats.20231120.sea_ad.MTG.h5` |

The Jakobsson samples are midbrain; Siletti covers the midbrain and has a dopaminergic (DA VGLUT2) cluster, so it is the default in `wf_config.py`. The file is several GB. The download URL below follows the [MapMyCells WHB-10Xv3 index](https://allen-brain-cell-atlas.s3.us-west-2.amazonaws.com/index.html#mapmycells/WHB-10Xv3/20240831/); check it there if the download fails.

In [ ]:
import subprocess

cfg.REFERENCES_PATH.mkdir(parents=True, exist_ok=True)
stats = cfg.MMC_PRECOMPUTED_STATS[cfg.MMC_TAXONOMY]
urls = {
    "Siletti": "https://allen-brain-cell-atlas.s3.us-west-2.amazonaws.com/mapmycells/WHB-10Xv3/20240831/precomputed_stats.siletti.training.h5",
}
if stats.exists():
    print(f"found {stats}")
elif cfg.MMC_TAXONOMY in urls:
    subprocess.run(["curl", "-fL", "-o", str(stats), urls[cfg.MMC_TAXONOMY]], check=True)
else:
    print(f"copy the {cfg.MMC_TAXONOMY} precomputed stats file to {stats}")

Setup is done. **Previous:** - · **Next:** [01_prep_metadata](01_prep_metadata.ipynb)